In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim

from torch.nn.utils.fusion import fuse_conv_bn_eval

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import onnx

import matplotlib.pyplot as plt
import numpy as np

from datetime import datetime
from time import time
import copy

In [2]:
import json_to_pytorch

In [3]:
def get_loaders(batch_size=64, num_workers=2, normalize=True):
    # Statistiche CIFAR-10 standard
    mean = (0.4914, 0.4822, 0.4465)
    std  = (0.2470, 0.2435, 0.2616)

    if normalize:
      train_tf = transforms.Compose([
          transforms.RandomCrop(32, padding=4),       # augmentation fondamentale
          transforms.RandomHorizontalFlip(),
          transforms.ToTensor(),
          transforms.Normalize(mean, std),
      ])
      test_tf = transforms.Compose([
          transforms.ToTensor(),
          transforms.Normalize(mean, std),
      ])
    else:
      train_tf = transforms.Compose([
          transforms.RandomCrop(32, padding=4),       # augmentation fondamentale
          transforms.RandomHorizontalFlip(),
          transforms.ToTensor(),
      ])
      test_tf = transforms.Compose([
          transforms.ToTensor(),
      ])

    train_ds = datasets.CIFAR10(root="./data", train=True,  download=True, transform=train_tf)
    test_ds  = datasets.CIFAR10(root="./data", train=False, download=True, transform=test_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    return train_loader, test_loader

In [4]:
criterion = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)

        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)

    return total_loss / total, correct / total

In [5]:
model = json_to_pytorch.json_to_pytorch("../VeryDiffPolyExperiments/results/gelu/best_model_bn_8_0.0001l1_no_pad_50.json", double_precision=True)

In [6]:
model

Sequential(
  (0): FrozenBatchNorm()
  (1): Conv2d(3, 8, kernel_size=(3, 3), stride=(2, 2))
  (2): ChebyshevPoly()
  (3): Conv2d(8, 16, kernel_size=(3, 3), stride=(2, 2))
  (4): ChebyshevPoly()
  (5): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2))
  (6): ChebyshevPoly()
  (7): Flatten(start_dim=1, end_dim=-1)
  (8): Linear(in_features=288, out_features=256, bias=True)
  (9): ChebyshevPoly()
  (10): Linear(in_features=256, out_features=10, bias=True)
)

In [7]:
train_loader, test_loader = get_loaders(batch_size=128)
test_loss,  test_acc  = evaluate(model, test_loader, criterion, "cpu")

Files already downloaded and verified
Files already downloaded and verified


In [8]:
test_acc

0.1004

In [9]:
# Get sample from test_loader
test_input, test_label = next(iter(test_loader))

In [10]:
test_input = torch.rand(test_input[0:1,:,:,:].shape)

In [11]:
torch.set_printoptions(threshold=30_000)

In [12]:
test_input

tensor([[[[1.7464e-01, 2.6444e-01, 4.1831e-01, 2.3265e-01, 5.6444e-01,
           5.6385e-01, 7.0133e-01, 9.9757e-01, 4.9917e-01, 4.4517e-01,
           5.9576e-01, 5.1895e-01, 2.1116e-01, 6.6672e-01, 6.2826e-01,
           4.2433e-01, 6.3703e-01, 3.3418e-01, 5.9394e-01, 5.3996e-01,
           6.3670e-01, 3.8758e-01, 2.8029e-01, 1.9691e-01, 1.1604e-02,
           5.2892e-01, 1.7027e-01, 6.4979e-01, 3.2948e-01, 8.4226e-01,
           5.0777e-01, 9.7011e-01],
          [9.6294e-01, 6.5093e-01, 7.2472e-01, 5.3610e-01, 7.2186e-01,
           8.8285e-01, 5.4199e-01, 4.0590e-01, 7.3724e-01, 4.9130e-01,
           1.8181e-01, 9.6203e-01, 2.0103e-01, 7.2621e-01, 9.4586e-01,
           9.1581e-01, 8.1658e-01, 4.3986e-01, 4.2634e-02, 3.9717e-01,
           5.3444e-01, 5.2206e-01, 3.6846e-02, 2.3080e-01, 5.4504e-01,
           3.4546e-01, 5.8783e-02, 5.6350e-01, 3.4738e-01, 8.5817e-02,
           3.1341e-01, 6.4144e-01],
          [5.5186e-01, 7.5129e-01, 5.9272e-01, 3.3951e-01, 4.4585e-01,
     

In [13]:
module_list = list(model.children())
module_list

[FrozenBatchNorm(),
 Conv2d(3, 8, kernel_size=(3, 3), stride=(2, 2)),
 ChebyshevPoly(),
 Conv2d(8, 16, kernel_size=(3, 3), stride=(2, 2)),
 ChebyshevPoly(),
 Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2)),
 ChebyshevPoly(),
 Flatten(start_dim=1, end_dim=-1),
 Linear(in_features=288, out_features=256, bias=True),
 ChebyshevPoly(),
 Linear(in_features=256, out_features=10, bias=True)]

In [14]:
model_part = nn.Sequential(*module_list[0:5])

In [15]:
model_part

Sequential(
  (0): FrozenBatchNorm()
  (1): Conv2d(3, 8, kernel_size=(3, 3), stride=(2, 2))
  (2): ChebyshevPoly()
  (3): Conv2d(8, 16, kernel_size=(3, 3), stride=(2, 2))
  (4): ChebyshevPoly()
)

In [16]:
model(test_input[0:1])

tensor([[ 2.8674,  8.6854,  3.7793, -5.0060, -6.0810, -7.7560,  1.8543,  1.2706,
         -0.5200,  2.0712]], dtype=torch.float64, grad_fn=<AddmmBackward0>)

In [17]:
import attack

In [32]:
adv = attack.Adversary()
adv_example, true_label, adv_label = adv.generate(model, test_input, 2.0/255.0, attack_iters=20, attack_alpha=1.0, attack_restarts=20)

In [33]:
print(f"True label: {true_label}, Adversarial label: {adv_label}")

True label: 1, Adversarial label: 1


In [34]:
model(test_input)

tensor([[ 2.8674,  8.6854,  3.7793, -5.0060, -6.0810, -7.7560,  1.8543,  1.2706,
         -0.5200,  2.0712]], dtype=torch.float64, grad_fn=<AddmmBackward0>)

In [31]:
model(adv_example)

tensor([[ 3.6766e+00,  6.8512e+00,  5.1585e+00, -4.7826e+00, -5.3460e+00,
         -8.2965e+00,  2.5734e+00,  9.8863e-01, -5.1266e-03,  3.5150e-01]],
       dtype=torch.float64, grad_fn=<AddmmBackward0>)